# Semantic Kernel with OpenBnB MCP Server Integration (C#)

This notebook demonstrates how to use Semantic Kernel in C# with the actual OpenBnB MCP server to search for real Airbnb accommodations using MCP plugins. For LLM Access, it uses Azure AI Foundry. To setup your environment variables, you can follow the Setup Lesson.

## Import the Needed Packages

In [ ]:
#r "nuget: Microsoft.SemanticKernel, 1.38.0"
#r "nuget: Microsoft.SemanticKernel.Agents.Core, 1.38.0-alpha"
#r "nuget: Microsoft.SemanticKernel.Connectors.MCP, 1.38.0-alpha"
#r "nuget: Azure.Identity, 1.13.1"

In [ ]:
// Import required namespaces
using System;
using System.Linq;
using System.Threading.Tasks;
using System.Collections.Generic;
using Azure.Identity;
using Microsoft.SemanticKernel;
using Microsoft.SemanticKernel.Agents;
using Microsoft.SemanticKernel.ChatCompletion;
using Microsoft.SemanticKernel.Connectors.AzureOpenAI;
using Microsoft.SemanticKernel.Connectors.MCP;

## Creating the MCP Plugin Connection

We'll connect to the [OpenBnB MCP server](https://github.com/openbnb-org/mcp-server-airbnb) using MCP Stdio Plugin. This server provides Airbnb search functionality through the @openbnb/mcp-server-airbnb package.

## Creating the Client

In this sample, we will use Azure AI Foundry for our LLM access. Make sure your environment variables are set up correctly.

## Environment Configuration

Configure Azure OpenAI settings. Make sure you have the following environment variables set:
- `AZURE_OPENAI_CHAT_DEPLOYMENT_NAME`
- `AZURE_OPENAI_ENDPOINT`
- `AZURE_OPENAI_API_KEY`

In [ ]:
// Load environment variables
var deploymentName = Environment.GetEnvironmentVariable("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME");
var endpoint = Environment.GetEnvironmentVariable("AZURE_OPENAI_ENDPOINT");
var apiKey = Environment.GetEnvironmentVariable("AZURE_OPENAI_API_KEY");

Console.WriteLine("🔍 Checking Azure environment variables...");
Console.WriteLine($"✅ AZURE_OPENAI_CHAT_DEPLOYMENT_NAME: {(!string.IsNullOrEmpty(deploymentName) ? "Set" : "NOT set")}");
Console.WriteLine($"✅ AZURE_OPENAI_ENDPOINT: {(!string.IsNullOrEmpty(endpoint) ? "Set" : "NOT set")}");
Console.WriteLine($"✅ AZURE_OPENAI_API_KEY: {(!string.IsNullOrEmpty(apiKey) ? "Set" : "NOT set")}");

if (string.IsNullOrEmpty(deploymentName) || string.IsNullOrEmpty(endpoint))
{
    throw new InvalidOperationException("Required environment variables are not set.");
}

## Understanding the OpenBnB MCP Integration

This notebook connects to the **real OpenBnB MCP server** that provides actual Airbnb search functionality.

### How it works:

1. **MCP Stdio Plugin**: Uses standard input/output communication with the MCP server
2. **Real NPM Package**: Downloads and runs `@openbnb/mcp-server-airbnb` via npx
3. **Live Data**: Returns actual Airbnb property data from their APIs
4. **Function Discovery**: The agent automatically discovers available functions from the MCP server

### Available Functions:

The OpenBnB MCP server typically provides:
- **search_listings** - Search for Airbnb properties by location and criteria
- **get_listing_details** - Get detailed information about specific properties
- **check_availability** - Check availability for specific dates
- **get_reviews** - Retrieve reviews for properties
- **get_host_info** - Get information about property hosts

### Prerequisites:

- **Node.js** installed on your system
- **Internet connection** to download the MCP server package
- **NPX** available (comes with Node.js)

### Testing the Connection:

You can test the MCP server manually by running:
```bash
npx -y @openbnb/mcp-server-airbnb
```

This will download and start the OpenBnB MCP server, which Semantic Kernel then connects to for real Airbnb data.

## Running the Agent with OpenBnB MCP Server

Now we will run the AI Agent that connects to the OpenBnB MCP server to search for real Airbnb accommodations in Stockholm for 2 adults and 1 kid. Feel free to change the user input to modify the search criteria.

In [ ]:
Console.WriteLine("🚀 Starting with Azure OpenAI...\n");

// Create kernel with Azure OpenAI
var kernelBuilder = Kernel.CreateBuilder();
kernelBuilder.AddAzureOpenAIChatCompletion(
    deploymentName: deploymentName,
    endpoint: endpoint,
    apiKey: apiKey
);

Console.WriteLine("🔧 Creating MCP Plugin...");

// Create MCP plugin connection to real OpenBnB server
var mcpPlugin = await MCPStdioPlugin.CreateAsync(
    name: "AirbnbSearch",
    description: "Search for Airbnb accommodations using OpenBnB MCP server",
    command: "npx",
    arguments: new[] { "-y", "@openbnb/mcp-server-airbnb" }
);

Console.WriteLine("✅ MCP Plugin created and connected");

// Add plugin to kernel
kernelBuilder.Plugins.Add(mcpPlugin);
var kernel = kernelBuilder.Build();

// Wait a moment for the server to fully initialize
await Task.Delay(2000);

// Try to list available tools
try
{
    var functions = kernel.Plugins.SelectMany(p => p).ToList();
    Console.WriteLine($"🔧 Available functions: {string.Join(", ", functions.Select(f => f.Name))}");
}
catch (Exception e)
{
    Console.WriteLine($"⚠️ Could not list tools: {e.Message}");
}

In [ ]:
// Create agent with Semantic Kernel
Console.WriteLine("\n🤖 Creating AI Agent...");

var agent = new ChatCompletionAgent
{
    Name = "AirbnbAgent",
    Instructions = @"You are an Airbnb search assistant. Use the available functions to search for properties. 
    Format results in a clear HTML table with columns for property name, price, rating, and link.",
    Kernel = kernel
};

Console.WriteLine("✅ Agent created with Azure OpenAI");

In [ ]:
// Execute agent with user request
var userInput = "Find Airbnb in Stockholm for 2 adults 1 kid";
Console.WriteLine($"\n🔍 User: {userInput}");

try
{
    var chatHistory = new ChatHistory();
    chatHistory.AddUserMessage(userInput);
    
    Console.WriteLine("\n🤖 Agent is processing your request...\n");
    
    await foreach (var message in agent.InvokeAsync(chatHistory))
    {
        var content = message.Content ?? string.Empty;
        
        // Remove markdown code blocks if present
        content = content.Replace("```html", "").Replace("```", "");
        
        Console.WriteLine($"🤖 {message.AuthorName}: ");
        
        if (content.Contains("<table", StringComparison.OrdinalIgnoreCase))
        {
            Console.WriteLine("[HTML Table Response - displayed in notebook output]");
            // In a Jupyter notebook, you can use display(HTML()) equivalent
            // For now, we'll show the raw HTML
            Console.WriteLine(content.Length > 500 ? content.Substring(0, 500) + "..." : content);
        }
        else
        {
            Console.WriteLine(content);
        }
        
        chatHistory.Add(message);
    }
    
    Console.WriteLine("\n✅ Request completed successfully!");
}
catch (Exception e)
{
    Console.WriteLine($"❌ Error processing user input: {e.Message}");
    Console.WriteLine(e.StackTrace);
}
finally
{
    // Cleanup MCP plugin
    if (mcpPlugin != null)
    {
        await mcpPlugin.DisposeAsync();
        Console.WriteLine("🧹 MCP Plugin cleaned up");
    }
}

# Summary

Congratulations! You've successfully built an AI agent in C# that integrates with real-world accommodation search using the Model Context Protocol (MCP):

## Technologies Used:
- **Semantic Kernel (C#)** - For building intelligent agents with Azure OpenAI
- **Azure AI Foundry** - For LLM capabilities and chat completion
- **MCP (Model Context Protocol)** - For standardized tool integration
- **OpenBnB MCP Server** - For real Airbnb search functionality
- **Node.js/NPX** - For running the external MCP server

## What You've Learned:
- **MCP Integration**: Connecting Semantic Kernel agents to external MCP servers in C#
- **Real-time Data Access**: Searching actual Airbnb properties through live APIs
- **Protocol Communication**: Using stdio communication between agent and MCP server
- **Function Discovery**: Automatically discovering available functions from MCP servers
- **Agent Pattern**: Building conversational agents with tool-calling capabilities
- **Error Handling**: Proper resource cleanup and exception management

## Next Steps:
- Integrate additional MCP servers (weather, flights, restaurants)
- Build a multi-agent system combining MCP and agent-to-agent protocols
- Create custom MCP servers for your own data sources
- Implement persistent conversation memory across sessions
- Deploy the agent to Azure Functions with MCP server orchestration
- Add user authentication and booking capabilities